===========================================================  
**Title: BCR Heavy-Chain Clonotypye Assignment**

**POC: Dr. Ian York**  
Team Lead, Molecular Virology and Vaccines Team  
Immunology and Pathogenesis Branch   
Influenza Division, NCIRD, OID   
Centers for Disease Control and Prevention   
**Email:** ite1@cdc.gov  

Other Contacts: Sujatha Seenu  
**Email:** iun1@cdc.gov  
============================================================
#### Overview

This workflow identifies clonotypes from BCR heavy-chain repertoire data using a network-based clustering approach. Sequences are grouped by **V gene**, **J gene**, and **CDR3 amino acid sequence length**, and clonotypes are assigned based on **≥80% CDR3 amino acid sequence identity**.

In [1]:
### Python libraries
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
from Bio import SeqIO, Seq
import itertools
from itertools import combinations

### Import or Install networkx python library
import networkx as nx  ### pip install networkx



In [2]:

#Input Validation and Data Loading
def load_input_data(prepared_dataframe=None, in_file_path=None, out_file_path=None):
 
    if out_file_path is not None:
        out_file_path = Path(out_file_path)
        if not out_file_path.parent.exists():
            raise FileNotFoundError(f'Output directory does not exist: {out_file_path.parent}')
 
    if prepared_dataframe is not None:
        if not isinstance(prepared_dataframe, pd.DataFrame):
            raise TypeError("'prepared_dataframe' must be a pandas DataFrame")
        return prepared_dataframe.copy()
 
    if in_file_path is None:
        raise ValueError("Either 'prepared_dataframe' or 'in_file_path' must be provided")
 
    in_file_path = Path(in_file_path)
    if not in_file_path.exists():
        raise FileNotFoundError(f'Input file not found: {in_file_path}')
 
    suffix = in_file_path.suffix.lower()
    if suffix == '.tsv':
        return pd.read_csv(in_file_path, sep='\t', low_memory=False)
    
    if suffix == '.xlsx':
        return pd.read_excel(in_file_path)
 
    raise ValueError("File extension isn't .tsv, .csv, or .xlsx")


def check_input(df):
    # Required column names
    req_fields = ['cdr3_aa', 'v_call', 'j_call']
    cols = df.columns
 
    missing = [field for field in req_fields if field not in cols]
    # Adjust required fields when junction_aa is provided instead of cdr3_aa
    if 'cdr3_aa' in missing and 'junction_aa' in df.columns: 
        missing.remove("cdr3_aa")
        req_fields.remove("cdr3_aa")
    if missing:
        raise ValueError('Missing required columns: %s' % ', '.join(missing))
        
    # Validate that all sequences in the input are BCR heavy-chain sequences.
    if len(df['v_call'].astype(str).str[:4].unique()) != 1 or df['v_call'].astype(str).str[:4].unique()[0] != 'IGHV':
        raise ValueError("Input must contain only heavy chain sequences")    
 
    # Need either junction_aa or cdr3_aa
    if 'junction_aa' not in cols and 'cdr3_aa' not in cols:
        raise ValueError("Input must contain either 'junction_aa' or 'cdr3_aa'")
 
    # Reject ANY NaN in required columns
    if df[req_fields].isna().any().any():
        bad = df[req_fields].isna().any()
        bad_cols = bad[bad].index.tolist()
        raise ValueError(f"NaN values found in required columns: {', '.join(bad_cols)}")
 
    # Reject empty strings in required string columns
    for col in ['v_call', 'j_call']:
        if df[col].astype('string').str.strip().eq('').any():
            raise ValueError(f"Column '{col}' contains empty strings")
 
     
    print('All required fields are present and valid in the input data')
    
    
# Load data
#===========
# Base directory
pth = ""
"""
IN_FILE_PATH: Path to the input file.

Supported input formats:
    - AIRR (Adaptive Immune Receptor Repertoire) TSV file
    - Excel file (.xlsx or .xls)
    - TSV file containing the required columns for BCR heavy chain listed below
"""

IN_FILE_PATH = Path(pth, "BCR_heavy_chain_input.tsv") # sample file
# out_file_path is a tsv 
OUT_FILE_PATH = Path(pth, "BCR_heavy_chain_clonotypes.tsv")

df = load_input_data(
    in_file_path=IN_FILE_PATH,
    out_file_path=OUT_FILE_PATH
)

# Validate input
check_input(df)

All required fields are present and valid in the input data


In [3]:
# Functions for Network-Based BCR Clonotype Identification

def sim_score(a,b):
    """ Calculate pairwise identity score for CDR3 sequences of same length """
    return sum([1 if a[i]==b[i] else 0 for i in range(len(a))])/len(a)


def find_families(cdr3s):
    """ Take a list of cdr3s and group into families
      Using itertools.combinations  https://docs.python.org/3/library/itertools.html#itertools.combinations """
    edges = []
    for i,j in itertools.combinations(range(len(cdr3s)),2):
        if sim_score(cdr3s[i], cdr3s[j]) >= 0.80: ## Apply cdr3 sequence identity cut-off
            edges.append((i,j))
    G = nx.Graph()
    G.add_nodes_from(range(len(cdr3s)))
    G.add_edges_from(edges)
    families = []
    for c in nx.connected_components(G):
        families.append([cdr3s[i] for i in c])
    return families


def same_cdr3_igh_family(df):
    """ Within each group with same HC  same junction length, use networkx to find those with 80% identity.
        ... for HC cdr3 ... """
    hc_jcts = df['cdr3_aa'].unique()
    cdr3_igh_families = []
    for cdr3_family in find_families(hc_jcts):
        cdr3_igh_families.append (df[df['cdr3_aa'].isin(cdr3_family)])
    return cdr3_igh_families


In [4]:
# Data Processing 

# Get the first assigned V and J gene calls
df['VH']=df['v_call'].apply(lambda x: str(x).split('*')[0] if x else None)
df['JH']=df['j_call'].apply(lambda x: str(x).split('*')[0] if x else None)

#combine V and J gene annotations
df['VJ_call']=df['VH']+"_"+df['JH']

# If cdr3_aa is not available, derive CDR3 amino acid sequences from
# junction_aa by removing the conserved N-terminal C and the conserved
# C-terminal residue (typically W or F).
if not "cdr3_aa" in df.columns:
    df["cdr3_aa"] = df["junction_aa"].astype(str).str[1:-1]

# Calculate the length of CDR3 amino acid sequences
df["cdr3_aa_length"]=df["cdr3_aa"].astype(str).str.len()  
    
clonotype_no=0

# Group BCR sequences by V gene, J gene, and junction length
grouped_bcrs = df.groupby(["VJ_call", "cdr3_aa_length"])
# Convert each groups into a DataFrame and store them in a dictionary
grouped_bcr_dataframes = {name: group.reset_index(drop=True) for name, group in grouped_bcrs}

#Assign clonotypes using a network-based clustering approach
for name, group_df in grouped_bcr_dataframes.items():
    for cdr3_igh_family in same_cdr3_igh_family(group_df):  
        family_name="clonotype_%d" % clonotype_no
        df_group=cdr3_igh_family.copy()
        df_group['clonotype_family']=family_name
            
        if clonotype_no == 0: # Include header
            df_group.to_csv(OUT_FILE_PATH, sep='\t',mode='w', index=False, header=True)  
        else:
            df_group.to_csv(OUT_FILE_PATH, sep='\t',mode='a', index=False, header=False) # Exclude header
            
        clonotype_no += 1
   
    